# Matched single-trial TW × alpha-decoding correlations and plots

This analysis matches TW and decoding rows using the saved original trial IDs and computes trial-wise TW × decoding correlations. `POOL_RAW_TRIALS_ACROSS_SESSIONS=True` directly concatenates both sessions' filtered raw trials before computing one subject-level r map; `False` computes two session-level maps and averages them. `FISHER_TRANSFORM` optionally applies Fisher z before group inference. The 19 subject maps are tested against 0 with a two-sided 2-D sign-flip cluster permutation.

In [10]:
from pathlib import Path
import importlib
import sys

PROJECT_DIR = Path('/home/dilay/project2/tw')
MODULE_DIR = PROJECT_DIR / 'travelling_waves/tw/fft/micheal/corr'
sys.path.insert(0, str(MODULE_DIR))

import michael_trial_correlation as trial_corr
trial_corr = importlib.reload(trial_corr)

build_subject_trial_maps = trial_corr.build_subject_trial_maps
cluster_signflip_trial_maps = trial_corr.cluster_signflip_trial_maps
plot_trial_maps = trial_corr.plot_trial_maps

TW_DIR = PROJECT_DIR / 'results/micheal_fft'
DECODING_DIR = PROJECT_DIR / 'results/michael_alpha_decoding_basis_kfold_fir_sliding_window_separate_sessions_8fold_100reps'

MEASURES = ('fw', 'bw')
COMPONENTS = ('difference', 'contra', 'ipsi', 'midline', 'all_lines')
FMIN, FMAX = 8, 12
TW_TMIN = 0.0
DECODING_TMIN = 0.0
FISHER_TRANSFORM = False  # True: Fisher z; False: raw Pearson r
POOL_RAW_TRIALS_ACROSS_SESSIONS = False
CORRELATION_MODE_TAG = (
    'raw-pooled' if POOL_RAW_TRIALS_ACROSS_SESSIONS else 'sessionwise'
)
CORRELATION_VALUE_LABEL = (
    'Mean within-subject Fisher z' if FISHER_TRANSFORM
    else 'Mean within-subject trial correlation (r)'
)
N_PERMUTATIONS = 5000
CLUSTER_ALPHA = 0.05
CLUSTER_P = 0.05
REPORT_CLUSTER_P = 0.10  # Print p < .10; contours/bars remain p < .05
SEED = 42

## Required new decoding fields

The alpha decoder must first be rerun. Every subject file must contain `sess1_trial_ids`, `sess1_trial_dec_early`, `sess1_trial_dec_late`, and the corresponding session-2 fields. Old subject-average-only files are rejected.

In [ ]:
# Build within-session correlations from exactly matched trial rows.
subjects, tw_time, decoding_time, subject_r, matched_counts = build_subject_trial_maps(
    TW_DIR,
    DECODING_DIR,
    measures=MEASURES,
    components=COMPONENTS,
    fmin=FMIN,
    fmax=FMAX,
    tw_tmin=TW_TMIN,
    decoding_tmin=DECODING_TMIN,
    fisher_transform=FISHER_TRANSFORM,
    pool_raw_trials_across_sessions=POOL_RAW_TRIALS_ACROSS_SESSIONS,
)

print('Subjects:', subjects)
print('TW time:', tw_time[[0, -1]])
print('Decoding time:', decoding_time[[0, -1]])
print('Matched trial counts:', matched_counts)
{measure: {
    component: {key: values.shape for key, values in condition.items()}
    for component, condition in components.items()
} for measure, components in subject_r.items()}

In [ ]:
# Alpha-band travelling-wave power itself (not decoding correlations).
# Columns: contra/ipsi/midline; rows: FW/BW. Shading is SEM across subjects.
import pickle
import matplotlib.pyplot as plt
import numpy as np
from mne.stats import permutation_cluster_1samp_test
from scipy.stats import t as student_t

TW_COMPONENTS = ('contra', 'ipsi', 'midline','all_lines')
TW_MEASURES = ('fw', 'bw')
EVENT_TIMES = (0.0, 1.2, 1.8, 3.8, 4.3)
EVENT_LABELS = ('target', 'impulse 1', 'rotation', 'impulse 2', 'probe')
STIM_WIDTHS = (0.30, 0.15, 0.30, 0.15, 0.30)

subject_tw = {
    measure: {component: [] for component in TW_COMPONENTS}
    for measure in TW_MEASURES
}
plot_time = None

for subject in subjects:
    session_tw = {
        measure: {component: [] for component in TW_COMPONENTS}
        for measure in TW_MEASURES
    }
    for session in (1, 2):
        tw_path = TW_DIR / f'subj{subject:02d}_sess{session}.pkl'
        with tw_path.open('rb') as stream:
            tw_data = pickle.load(stream)

        current_time = trial_corr._tw_time(tw_data)
        time_mask = current_time >= TW_TMIN
        if plot_time is None:
            plot_time = current_time[time_mask]
        elif not np.allclose(plot_time, current_time[time_mask]):
            raise ValueError(f'TW time mismatch in {tw_path.name}')

        frequencies = np.asarray(tw_data['ff'])
        frequency_mask = (frequencies >= FMIN) & (frequencies <= FMAX)
        keep = np.ones(np.asarray(tw_data['cue_loc']).size, dtype=bool)
        if 'is_bad_epoch' in tw_data:
            keep &= ~np.asarray(tw_data['is_bad_epoch'], dtype=bool)
        if 'has_timing_issue' in tw_data:
            keep &= ~np.asarray(tw_data['has_timing_issue'], dtype=bool)

        for measure in TW_MEASURES:
            values = trial_corr._tw_measure(tw_data, measure)
            components = trial_corr._component_arrays(
                values, np.asarray(tw_data['cue_loc'])
            )
            for component in TW_COMPONENTS:
                # component: frequency × TW-time × trial.
                time_course = components[component][frequency_mask][
                    :, time_mask, :
                ][:, :, keep].mean(axis=(0, 2))
                session_tw[measure][component].append(time_course)

    for measure in TW_MEASURES:
        for component in TW_COMPONENTS:
            subject_tw[measure][component].append(
                np.mean(session_tw[measure][component], axis=0)
            )

subject_tw = {
    measure: {
        component: np.stack(values)
        for component, values in components.items()
    }
    for measure, components in subject_tw.items()
}

def significant_time_clusters(values, seed):
    threshold = student_t.ppf(1 - CLUSTER_ALPHA / 2, values.shape[0] - 1)
    _, clusters, p_values, _ = permutation_cluster_1samp_test(
        values,
        threshold=threshold,
        n_permutations=N_PERMUTATIONS,
        tail=0,
        seed=seed,
        n_jobs=1,
        out_type='indices',
        verbose=False,
    )
    return [
        (np.asarray(cluster[0], dtype=int), float(p_value))
        for cluster, p_value in zip(clusters, p_values)
        if p_value < REPORT_CLUSTER_P
    ]

colors = {'fw': '#1f77b4', 'bw': '#ff7f0e'}
fig, axes = plt.subplots(
    len(TW_MEASURES), len(TW_COMPONENTS),
    figsize=(6.5 * len(TW_COMPONENTS), 7.5),
    sharex=True, constrained_layout=True,
)

for row, measure in enumerate(TW_MEASURES):
    for col, component in enumerate(TW_COMPONENTS):
        axis = axes[row, col]
        values = subject_tw[measure][component]
        mean = values.mean(axis=0)
        sem = values.std(axis=0, ddof=1) / np.sqrt(values.shape[0])
        clusters = significant_time_clusters(values, SEED + 20000 + 10 * row + col)

        axis.plot(plot_time, mean, color=colors[measure], linewidth=2.3)
        axis.fill_between(
            plot_time, mean - sem, mean + sem,
            color=colors[measure], alpha=0.22, linewidth=0,
        )
        axis.axhline(0, color='0.45', linewidth=0.8)
        for event, width, label in zip(EVENT_TIMES, STIM_WIDTHS, EVENT_LABELS):
            axis.axvline(event, color='0.45', linestyle='--', linewidth=0.8, alpha=0.6)
            axis.axvspan(event, event + width, ymin=0, ymax=0.035, color='0.4', alpha=0.75)
            if row == 1:
                axis.text(
                    event, -0.12, label, transform=axis.get_xaxis_transform(),
                    rotation=35, ha='right', va='top', fontsize=8,
                )

        ymin, ymax = axis.get_ylim()
        cluster_y = ymax - 0.05 * (ymax - ymin)
        for indices, p_value in clusters:
            status = 'significant' if p_value < CLUSTER_P else 'trend'
            if p_value < CLUSTER_P:
                axis.plot(
                    [plot_time[indices[0]], plot_time[indices[-1]]],
                    [cluster_y, cluster_y],
                    color='black', linewidth=4, solid_capstyle='butt',
                )
            print(
                f'{measure.upper()} {component}: '
                f'{plot_time[indices[0]]:.3f}–{plot_time[indices[-1]]:.3f} s, '
                f'cluster p={p_value:.5f} ({status})'
            )

        axis.set_title(f'{component.capitalize()} | Alpha (8–12 Hz)')
        axis.set_xlim(plot_time[0], plot_time[-1])
        axis.spines[['top', 'right']].set_visible(False)
        if row == 1:
            axis.set_xlabel('Time (s)')
        if col == 0:
            axis.set_ylabel(f'{measure.upper()} TW power (dB)')

fig.suptitle(f'Contra and ipsi alpha travelling-wave power (N={len(subjects)})', fontsize=15)
plt.show()

In [ ]:
# Group inference on 19 subject maps (Fisher z or raw r, selected above).
from IPython.display import Image, display

component_tag = {
    'difference': 'contra-minus-ipsi',
    'contra': 'contra-only',
    'ipsi': 'ipsi-only',
    'midline': 'midline',
    'all_lines': 'all-lines',
}
OUTPUTS = {}
STATISTICS = {}

for measure_index, measure in enumerate(MEASURES):
    for component_index, component in enumerate(COMPONENTS):
        maps, clusters = {}, {}
        print(f'\n=== trial correlation: {measure.upper()} / {component} ===')
        for condition_index, key in enumerate(('early', 'late')):
            maps[key], clusters[key] = cluster_signflip_trial_maps(
                subject_r[measure][component][key],
                permutations=N_PERMUTATIONS,
                cluster_alpha=CLUSTER_ALPHA,
                cluster_p=CLUSTER_P,
                seed=SEED + 1000*measure_index + 100*component_index + condition_index,
                return_all=True,
            )
            report_clusters = [
                cluster for cluster in clusters[key]
                if cluster['p'] < REPORT_CLUSTER_P
            ]
            significant_clusters = [
                cluster for cluster in clusters[key]
                if cluster['p'] < CLUSTER_P
            ]
            print(
                f'  tested {key}: {len(significant_clusters)} significant; '
                f'{len(report_clusters)} cluster(s) with p < {REPORT_CLUSTER_P:g}'
            )
            for cluster in sorted(report_clusters, key=lambda item: item['p']):
                status = 'significant' if cluster['p'] < CLUSTER_P else 'trend'
                print(
                    f"    TW {tw_time[cluster['row_min']]:.3f}–"
                    f"{tw_time[cluster['row_max']]:.3f} s; "
                    f"decoding {decoding_time[cluster['col_min']]*1000:.0f}–"
                    f"{decoding_time[cluster['col_max']]*1000:.0f} ms; "
                    f"cluster p={cluster['p']:.5f} ({status})"
                )
            clusters[key] = significant_clusters

        tag = component_tag[component]
        output = DECODING_DIR / f'trial-corr_{CORRELATION_MODE_TAG}_{measure}_alpha_{tag}_time0.png'
        title = (
            f'Matched-trial {measure.upper()} TW ({tag}) × alpha decoding'
        )
        plot_trial_maps(
            maps, clusters, decoding_time, tw_time, output, title,
            value_label=CORRELATION_VALUE_LABEL,
        )
        OUTPUTS[(measure, component)] = output
        STATISTICS[(measure, component)] = {'maps': maps, 'clusters': clusters}
        print('  saved:', output)
        display(Image(filename=str(output)))

OUTPUTS

In [ ]:
# Correlate FW and BW with the within-trial decoding difference:
# tested early decoding minus tested late decoding.
import matplotlib.pyplot as plt
import numpy as np

DIFFERENCE_OUTPUTS = {}
DIFFERENCE_STATISTICS = {}
DIFFERENCE_COMPONENTS = ('difference', 'contra', 'ipsi', 'midline', 'all_lines')
decoding_ms = decoding_time * 1000

for component_index, component in enumerate(DIFFERENCE_COMPONENTS):
    maps = {}
    clusters = {}

    print(f'\n{component}: tested early - tested late')
    for measure_index, measure in enumerate(('fw', 'bw')):
        maps[measure], clusters[measure] = cluster_signflip_trial_maps(
            subject_r[measure][component]['early_minus_late'],
            permutations=N_PERMUTATIONS,
            cluster_alpha=CLUSTER_ALPHA,
            cluster_p=CLUSTER_P,
            seed=SEED + 5000 + 100 * component_index + measure_index,
            return_all=True,
        )
        significant = [cluster for cluster in clusters[measure] if cluster['p'] < CLUSTER_P]
        report_clusters = [
            cluster for cluster in clusters[measure]
            if cluster['p'] < REPORT_CLUSTER_P
        ]
        print(
            f'  {measure.upper()}: {len(significant)} significant; '
            f'{len(report_clusters)} cluster(s) with p < {REPORT_CLUSTER_P:g}'
        )
        for cluster in sorted(report_clusters, key=lambda item: item['p']):
            status = 'significant' if cluster['p'] < CLUSTER_P else 'trend'
            print(
                f"    TW {tw_time[cluster['row_min']]:.3f}–"
                f"{tw_time[cluster['row_max']]:.3f} s; "
                f"decoding {decoding_ms[cluster['col_min']]:.0f}–"
                f"{decoding_ms[cluster['col_max']]:.0f} ms; "
                f"p={cluster['p']:.5f} ({status})"
            )
        clusters[measure] = significant

    limit = max(float(np.nanmax(np.abs(maps['fw']))), float(np.nanmax(np.abs(maps['bw']))))
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.8), sharex=True, sharey=True)
    image = None
    for axis, measure in zip(axes, ('fw', 'bw')):
        image = axis.imshow(
            maps[measure],
            origin='lower',
            aspect='auto',
            extent=[decoding_ms[0], decoding_ms[-1], tw_time[0], tw_time[-1]],
            cmap='RdBu_r',
            vmin=-limit,
            vmax=limit,
            interpolation='nearest',
        )
        for cluster in clusters[measure]:
            if cluster['p'] < CLUSTER_P:
                axis.contour(
                    decoding_ms,
                    tw_time,
                    cluster['mask'].astype(float),
                    levels=[0.5],
                    colors='black',
                    linewidths=2,
                )
        for event_time in trial_corr.EVENT_TIMES:
            axis.axhline(event_time, color='0.55', linestyle=':', linewidth=0.8)
            axis.axvline(event_time * 1000, color='0.55', linestyle=':', linewidth=0.8)
        axis.set_title(f"{measure.upper()} × (tested early − tested late)")
        axis.set_xlabel('Single-trial decoding time (ms)')
        axis.grid(False)

    axes[0].set_ylabel('Single-trial travelling-wave time (s)')
    component_label = component_tag[component]
    fig.suptitle(
        f'Matched-trial FW and BW TW ({component_label}) × decoding difference',
        fontsize=16,
    )
    fig.subplots_adjust(left=0.07, right=0.88, bottom=0.14, top=0.84, wspace=0.08)
    colorbar_axis = fig.add_axes([0.90, 0.18, 0.018, 0.60])
    colorbar = fig.colorbar(image, cax=colorbar_axis)
    colorbar.set_label(CORRELATION_VALUE_LABEL)

    tag = component_tag[component]
    output = DECODING_DIR / f'trial-corr_{CORRELATION_MODE_TAG}_fw-bw_alpha_{tag}_early-minus-late_time0.png'
    fig.savefig(output, dpi=180, bbox_inches='tight')
    plt.close(fig)

    DIFFERENCE_OUTPUTS[component] = output
    DIFFERENCE_STATISTICS[component] = {'maps': maps, 'clusters': clusters}
    print('  saved:', output)
    display(Image(filename=str(output)))

DIFFERENCE_OUTPUTS

## Segment-wise TW-time cluster tests

Each cell below crops **both TW time (y) and decoding time (x) to the same experimental segment**, then recomputes the complete 2D cluster-permutation null distribution inside that segment. Every segment automatically analyzes both `FW` and `BW`; for each direction it analyzes `ipsi`, `contra`, `difference (contra − ipsi)`, and `midline`; and for every combination it tests `dec_early`, `dec_late`, and the trial-wise difference `dec_early − dec_late`. Segment-wise `p < .05` is shown with black contours. Because three segments are examined, the stricter Bonferroni threshold across segments is `.05 / 3 = .0167`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Reload the plotting module so ROI axis-limit fixes take effect even when
# this notebook kernel imported an older version earlier.
trial_corr = importlib.reload(trial_corr)
plot_trial_maps = trial_corr.plot_trial_maps
cluster_signflip_trial_maps = trial_corr.cluster_signflip_trial_maps

ROI_MEASURES = ('fw', 'bw')
ROI_COMPONENTS = ('ipsi', 'contra', 'difference', 'midline', 'all_lines')
SEGMENT_CLUSTER_P = 0.05
ACROSS_SEGMENTS_P = 0.05 / 3

def run_tw_segment(tmin, tmax, segment_index, measure, component):
    row_mask = (tw_time >= tmin) & (tw_time <= tmax)
    column_mask = (decoding_time >= tmin) & (decoding_time <= tmax)
    segment_time = tw_time[row_mask]
    segment_decoding_time = decoding_time[column_mask]
    if not row_mask.any() or not column_mask.any():
        raise ValueError(f'No TW/decoding samples in {tmin}–{tmax} s')

    maps = {}
    significant_clusters = {}
    all_clusters = {}
    print(f'\n=== {measure.upper()} / {component}: TW {tmin:g}–{tmax:g} s ===')
    for condition_index, key in enumerate(('early', 'late', 'early_minus_late')):
        segment_subject_maps = subject_r[measure][component][key][
            :, row_mask, :
        ][:, :, column_mask]
        maps[key], all_clusters[key] = cluster_signflip_trial_maps(
            segment_subject_maps,
            permutations=N_PERMUTATIONS,
            cluster_alpha=CLUSTER_ALPHA,
            cluster_p=SEGMENT_CLUSTER_P,
            seed=SEED + 20000 + 100 * segment_index + condition_index,
            return_all=True,
        )
        significant_clusters[key] = [
            cluster for cluster in all_clusters[key]
            if cluster['p'] < SEGMENT_CLUSTER_P
        ]
        report_clusters = [
            cluster for cluster in all_clusters[key]
            if cluster['p'] < REPORT_CLUSTER_P
        ]
        print(
            f'  tested {key}: {len(significant_clusters[key])} significant '
            f'cluster(s); {len(report_clusters)} cluster(s) with p < {REPORT_CLUSTER_P:g}'
        )
        if not report_clusters:
            print(f'    no cluster with p < {REPORT_CLUSTER_P:g}')
            continue
        for cluster_index, cluster in enumerate(
            sorted(report_clusters, key=lambda item: item['p']), start=1
        ):
            status = 'significant' if cluster['p'] < SEGMENT_CLUSTER_P else 'trend'
            print(
                f"    cluster {cluster_index}: TW "
                f"{segment_time[cluster['row_min']]:.3f}–"
                f"{segment_time[cluster['row_max']]:.3f} s; decoding "
                f"{segment_decoding_time[cluster['col_min']]*1000:.0f}–"
                f"{segment_decoding_time[cluster['col_max']]*1000:.0f} ms; "
                f"segment-corrected p={cluster['p']:.5f} ({status}); "
                f"survives 3-segment correction="
                f"{cluster['p'] < ACROSS_SEGMENTS_P}"
            )

    tag = component_tag[component]
    segment_tag = f'{tmin:g}-{tmax:g}s'.replace('.', 'p')
    output = DECODING_DIR / (
        f'trial-corr_{CORRELATION_MODE_TAG}_{measure}_alpha_{tag}_tw-segment_{segment_tag}.png'
    )
    title = (
        f'Matched-trial {measure.upper()} TW ({tag}) × alpha decoding '
        f'| TW {tmin:g}–{tmax:g} s'
    )
    plot_trial_maps(
        maps, significant_clusters, segment_decoding_time, segment_time,
        output, title, value_label=CORRELATION_VALUE_LABEL
    )
    display(Image(filename=str(output)))

    # Additional map: TW correlation with trial-wise early − late decoding.
    difference_values = maps['early_minus_late']
    difference_output = DECODING_DIR / (
        f'trial-corr_{CORRELATION_MODE_TAG}_{measure}_alpha_{tag}_early-minus-late_'
        f'tw-segment_{segment_tag}.png'
    )
    difference_limit = float(np.nanmax(np.abs(difference_values)))
    fig, axis = plt.subplots(figsize=(7.6, 5.4))
    image = axis.imshow(
        difference_values,
        origin='lower',
        aspect='auto',
        extent=[
            segment_decoding_time[0] * 1000,
            segment_decoding_time[-1] * 1000,
            segment_time[0],
            segment_time[-1],
        ],
        cmap='RdBu_r',
        vmin=-difference_limit,
        vmax=difference_limit,
        interpolation='nearest',
    )
    for cluster in significant_clusters['early_minus_late']:
        axis.contour(
            segment_decoding_time * 1000,
            segment_time,
            cluster['mask'].astype(float),
            levels=[0.5],
            colors='black',
            linewidths=2,
        )
    for event_time in trial_corr.EVENT_TIMES:
        if tmin <= event_time <= tmax:
            axis.axhline(event_time, color='0.55', linestyle=':', linewidth=0.8)
            axis.axvline(event_time * 1000, color='0.55', linestyle=':', linewidth=0.8)
    axis.set_xlim(segment_decoding_time[0] * 1000, segment_decoding_time[-1] * 1000)
    axis.set_ylim(segment_time[0], segment_time[-1])
    axis.set_xlabel('Single-trial decoding-difference time (ms)')
    axis.set_ylabel('Single-trial travelling-wave time (s)')
    axis.set_title(
        f'{measure.upper()} TW ({tag}) × (tested early − tested late) '
        f'| {tmin:g}–{tmax:g} s'
    )
    colorbar = fig.colorbar(image, ax=axis)
    colorbar.set_label(CORRELATION_VALUE_LABEL)
    fig.tight_layout()
    fig.savefig(difference_output, dpi=180, bbox_inches='tight')
    plt.close(fig)
    display(Image(filename=str(difference_output)))
    return {
        'output': output,
        'difference_output': difference_output,
        'maps': maps,
        'clusters': all_clusters,
        'significant_clusters': significant_clusters,
    }

def run_all_components_segment(tmin, tmax, segment_index):
    results = {}
    for measure_index, measure in enumerate(ROI_MEASURES):
        results[measure] = {}
        for component_index, component in enumerate(ROI_COMPONENTS):
            results[measure][component] = run_tw_segment(
                tmin,
                tmax,
                segment_index=(100 * segment_index + 10 * measure_index + component_index),
                measure=measure,
                component=component,
            )
    return results


In [ ]:
# Segment 1: TW time 0–1.2 s
SEGMENT_0_1P2 = run_all_components_segment(0.0, 1.8, segment_index=1)


In [ ]:
# Segment 2: TW time 1.8–3.8 s
SEGMENT_1P8_3P8 = run_all_components_segment(1.8, 4.3, segment_index=2)


In [ ]:
# Segment 3: TW time 3.8 s to the end
SEGMENT_3P8_END = run_all_components_segment(
    3.8, float(tw_time[-1]), segment_index=3
)
